### Xgboost from PCA v3 dataset 

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import json
import os
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import random as rd
from datetime import datetime

In [ ]:
note_for_docs = "70% test, 30% train. 5 fold groupshufflesplit. PCA data."

In [ ]:
def log_experiment(params, accuracy, cm_normalized, labels, filepath="experiment_logs.jsonl", mode="jsonl"):
    """
    Logs hyperparameter tuning runs to either JSON Lines or CSV.
    
    cm_normalized: numpy array from confusion_matrix(y_test, y_pred, normalize='true')
    labels: list/array of class names from le.classes_
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    if mode == "jsonl":
        # Create dictionary representation of the confusion matrix (actual -> predicted -> %)
        cm_dict = {
            actual: {pred: round(float(cm_normalized[i, j] * 100), 2) 
                     for j, pred in enumerate(labels)}
            for i, actual in enumerate(labels)
        }
        
        entry = {
            "timestamp": timestamp,
            "accuracy": round(float(accuracy), 4),
            "hyperparameters": params,
            "confusion_matrix_pct": cm_dict,
            "note": note_for_docs
        }
        
        with open(filepath, "a") as f:
            f.write(json.dumps(entry) + "\n")
        print(f"Logged run to {filepath} (JSONL)")

    elif mode == "csv":
        # Flatten parameters and confusion matrix entries into a single dictionary row
        row = {
            "timestamp": timestamp,
            "accuracy": round(float(accuracy), 4),
            **{f"param_{k}": v for k, v in params.items()}
        }
        
        # Flatten confusion matrix percentages: cm_<actual>_<predicted>
        for i, actual in enumerate(labels):
            for j, pred in enumerate(labels):
                col_name = f"cm_{actual}_{pred}"
                row[col_name] = round(float(cm_normalized[i, j] * 100), 2)
        
        df_row = pd.DataFrame([row])
        file_exists = os.path.isfile(filepath)
        df_row.to_csv(filepath, mode="a", index=False, header=not file_exists)
        print(f"Logged run to {filepath} (CSV)")

In [ ]:
param_ = {
    'learning_rate': 0.01,
    'max_depth': 1,
    'n_estimators': 2,
    'num_round': 50,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 2
}
param = {"learning_rate": 0.1, "max_depth": 4, "n_estimators": 1000, "num_round": 50, "subsample": 0.8, "colsample_bytree": 0.8, "min_child_weight": 5}

In [ ]:
df_pca = pd.read_csv('../dataset/full_v4_novariancethreshold.csv')
df_pca.head()

In [ ]:
X = df_pca.drop(["Activity", "subject"], axis=1) # Predictors (69 components)
y = df_pca["Activity"]                        # Target variable
le = LabelEncoder()
y = le.fit_transform(y)
y = pd.DataFrame(y)
groups = df_pca["subject"]

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.7, random_state=69)

# Get indices for train and test sets based on Subject groups
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

train_subjects = sorted(groups_train.unique())
test_subjects = sorted(groups_test.unique())

print("Subjects in training data:")
print(train_subjects)

print("\nSubjects in test data:")
print(test_subjects)

print("\nNumber of training subjects:", len(train_subjects))
print("Number of test subjects:", len(test_subjects))

overlap = set(train_subjects) & set(test_subjects)
print("\nOverlapping subjects:", overlap)

In [ ]:
model = xgb.XGBClassifier(**param)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)



y_decoded = le.inverse_transform(y_test)
y_pred_decoded = le.inverse_transform(y_pred)
y_decoded = pd.DataFrame(y_decoded)
y_pred_decoded = pd.DataFrame(y_pred_decoded)

print("--- Classification Report (Unseen Subjects) ---")
print(classification_report(y_decoded, y_pred_decoded))

In [ ]:
# Compute normalized matrix (percentages along rows)
cm_normalized = confusion_matrix(y_test, y_pred, normalize='true')

# Compute metrics
accuracy = accuracy_score(y_test, y_pred)
cm_normalized = confusion_matrix(y_test, y_pred, normalize='true')
labels = le.classes_

# Log your experiment (Option 1: JSON Lines)
log_experiment(
    params=param,
    accuracy=accuracy,
    cm_normalized=cm_normalized,
    labels=labels,
    filepath="../dataset/experiment_logs.jsonl",
    mode="jsonl"
)

# Get activity labels in the same order used by LabelEncoder
labels = le.classes_

plt.figure(figsize=(10, 7))
sns.heatmap(
    cm_normalized, 
    annot=True, 
    fmt='.1%',              # Format as percentage (e.g., 70.2%)
    cmap='Blues', 
    xticklabels=labels, 
    yticklabels=labels
)

plt.title('Normalized Confusion Matrix (%)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('Actual Label (Ground Truth)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.show()